# Step 9: Statistical Analysis
For each test: Hypothesis -> Test -> Assumptions -> p-value -> Decision -> Business insight

In [ ]:
import pandas as pd
from scipy import stats

orders = pd.read_csv('../data/cleaned/orders_features.csv')
order_items = pd.read_csv('../data/cleaned/order_items.csv')
products = pd.read_csv('../data/cleaned/products.csv')
reviews = pd.read_csv('../data/cleaned/order_reviews.csv')
payments = pd.read_csv('../data/cleaned/order_payments.csv')

## 1. T-Test: Delivery Delay vs Review Score
**H0:** No difference in review scores between delayed and on-time orders.
**H1:** Delayed orders receive significantly lower review scores.
**Test:** Independent two-sample t-test (one-tailed)

In [ ]:
merged = orders.merge(reviews, on='order_id')

delayed_scores = merged.loc[merged['is_delayed'] == True, 'review_score'].dropna()
ontime_scores = merged.loc[merged['is_delayed'] == False, 'review_score'].dropna()

# Check variance assumption (Levene's test) before choosing equal_var
levene_stat, levene_p = stats.levene(delayed_scores, ontime_scores)
equal_var = levene_p > 0.05

t_stat, p_value = stats.ttest_ind(delayed_scores, ontime_scores, equal_var=equal_var, alternative='less')

print(f'Levene p-value: {levene_p:.4f} -> equal_var={equal_var}')
print(f'Delayed mean: {delayed_scores.mean():.2f}, On-time mean: {ontime_scores.mean():.2f}')
print(f't-statistic: {t_stat:.4f}, p-value: {p_value:.4f}')
print('Decision:', 'Reject H0 - delayed orders have significantly lower scores'
      if p_value < 0.05 else 'Fail to reject H0')

## 2. ANOVA: Order Value Across Product Categories
**H0:** Mean order value is equal across all product categories.
**H1:** At least one category differs significantly.
**Test:** One-Way ANOVA

In [ ]:
items_products = order_items.merge(products, on='product_id')

category_groups = [
    group['price'].values
    for _, group in items_products.groupby('product_category_name')
    if len(group) >= 30  # keep categories with enough sample size
]

f_stat, p_value = stats.f_oneway(*category_groups)
print(f'F-statistic: {f_stat:.4f}, p-value: {p_value:.4f}')
print('Decision:', 'Reject H0 - spending differs significantly across categories'
      if p_value < 0.05 else 'Fail to reject H0')

## 3. Chi-Square: Payment Method vs Order Status
**H0:** Payment method and order status are independent.
**H1:** Payment method and order status are associated.
**Test:** Chi-Square Test of Independence

In [ ]:
merged_pay = orders.merge(payments, on='order_id')
contingency = pd.crosstab(merged_pay['payment_type'], merged_pay['order_status'])

chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print(f'Chi2: {chi2:.4f}, dof: {dof}, p-value: {p_value:.4f}')
print('Decision:', 'Reject H0 - payment method and order status are associated'
      if p_value < 0.05 else 'Fail to reject H0')